# Chapter 01 — The Loop You Are Writing For

**Companion to *Pi Agents*** · [`Pi Agents` chapter 01](https://programmer.ie/books/pi/01-chapter/)

| | |
|---|---|
| Chapter | `01-chapter.md` · weight 1 · `/books/pi/01-chapter/` |
| Notebook | `notebooks/pi/01-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

One turn, with no `pi` binary, no terminal and no credential: what actually
happens, and **where in the stream is the decision "is anything else needed?"**

The chapter opens by taking everything away — no binary, no interface, no
credential — and then puts back the one loop that remains. The interesting
question is not what the loop does but *what ends it*, because that is the
thing you have to be able to see in order to control it.


## What this notebook establishes

- A single prompt produces an ordered event stream beginning with `agent_start` and ending with `agent_end`.
- The loop continues **exactly** while the assistant's `stopReason` is `toolUse`, and stops on any other `stopReason`.
- One turn costs a countable number of provider requests: 2 for tool-then-answer, 3 when the model asks twice, 1 when the provider fails on the first call.
- The bare agent core does **not** emit `agent_settled`. That event belongs to the coding agent above the core, and a program reading the core's stream will not find it.

## What this notebook does **not** establish

- Nothing about a real model. The provider is scripted: every case exercised here is a case someone anticipated.
- Nothing about token accounting, retries or provider internals — the core has no notion of them (chapter 29).
- Nothing about how often a real model keeps asking for tools. The three-request case below is a script, not a measurement.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's own example, then a structured capture of the same loop.
* **Evidence scope:** `in_process_runtime` (`@earendil-works/pi-agent-core` only). No binary, no interface.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(1))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/01-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the chapter's example, run unmodified

`examples/ch01-agent-core/agent-core.ts` is the file the chapter embeds verbatim.
It is 45 lines: a tool, two scripted responses, an `Agent`, and a subscriber that
prints two kinds of event. Running it is the smallest honest evidence that the
loop the book describes is the loop that exists.

In [3]:
r = pinb.run_node(["ch01-agent-core/agent-core.ts"], timeout=60)
print(r.assert_ok('the chapter 1 example').describe())

$ C:\Program Files\nodejs\node.EXE ch01-agent-core/agent-core.ts
  exit=0 in 320 ms
--- stdout ---
stop: toolUse
tool: word_count {"words":4}
stop: stop


Three lines, and each one is a mechanism:

* `stop: toolUse` — the assistant asked for a tool, so the loop continues.
* `tool: word_count {"words":4}` — the tool's `execute()` ran and returned `details`.
* `stop: stop` — the assistant answered with no tool call, so the loop ends.

The example prints. What a program needs is the *sequence*, so the next cell runs
the same loop and records it.

## Experiment: make the stop decision queryable

`drivers/ch01-loop.ts` builds the same agent core the chapter builds — same tool,
same provider, same `Agent` constructor — and subscribes to the stream to record the
ordered event types plus each assistant message's `stopReason`.

This is orchestration, not re-implementation: it calls
`@earendil-works/pi-agent-core` and substitutes nothing except the model.

In [4]:
DRIVER = pinb.driver_source("ch01-loop.ts")
runs = pinb.driver(DRIVER, label="ch01-loop", timeout=180)
print(pinb.table(
    ["run", "requests", "assistant stopReasons", "tool runs"],
    [[name.replace(chr(95), ' '), r['requests'], ' -> '.join(r['stopReasons']), len(r['toolRuns'])]
     for name, r in runs.items()],
    indent="",
))

run  requests  assistant stopReasons  tool runs
-----------------  --------  --------------------------  ---------
oneToolThenAnswer  2  toolUse -> stop  1
keepsAsking  3  toolUse -> toolUse -> stop  2
providerError  1  error  0


In [5]:
for name, r in runs.items():
    print(f"\n{name}:")
    for i, ev in enumerate(r["stream"], 1):
        print(f"  {i:2d}. {ev}")


oneToolThenAnswer:
   1. agent_start
   2. turn_start
   3. message_start
   4. message_end
   5. message_start
   6. message_update
   7. message_update
   8. message_update
   9. message_update
  10. message_end:toolUse
  11. tool_execution_start:word_count
  12. tool_execution_end:word_count
  13. message_start
  14. message_end
  15. turn_end
  16. turn_start
  17. message_start
  18. message_update
  19. message_update
  20. message_update
  21. message_end:stop
  22. turn_end
  23. agent_end

keepsAsking:
   1. agent_start
   2. turn_start
   3. message_start
   4. message_end
   5. message_start
   6. message_update
   7. message_update
   8. message_update
   9. message_end:toolUse
  10. tool_execution_start:word_count
  11. tool_execution_end:word_count
  12. message_start
  13. message_end
  14. turn_end
  15. turn_start
  16. message_start
  17. message_update
  18. message_update
  19. message_update
  20. message_end:toolUse
  21. tool_execution_start:word_count
  22. too

## The assertion that matters

Three facts, each checked against a boundary rather than against a string the
script just wrote:

* the request count, which is the only cost the loop imposes;
* the assistant's `stopReason` sequence, which *is* the termination decision;
* the absence of `agent_settled` from the core's own stream — a positive claim
  about what the core does not emit.

In [6]:
one = runs["oneToolThenAnswer"]
many = runs["keepsAsking"]
bad = runs["providerError"]

pinb.show_checks([
    pinb.check("the stream starts at agent_start", one["stream"][0] == "agent_start", one["stream"][0]),
    pinb.check("tool-then-answer costs exactly two provider requests", one["requests"] == 2, str(one["requests"])),
    pinb.check("the loop continued exactly while stopReason was toolUse",
               one["stopReasons"] == ["toolUse", "stop"], str(one["stopReasons"])),
    pinb.check("the tool body ran once, and its details reached the stream",
               one["toolRuns"] == [{"words": 4}], str(one["toolRuns"])),
    pinb.check("a model that keeps asking costs one request per ask",
               many["requests"] == 3, str(many["requests"])),
    pinb.check("a provider failure costs one request and runs no tool",
               bad["requests"] == 1 and bad["toolRuns"] == [],
               f"requests={bad['requests']} tools={len(bad['toolRuns'])}"),
    pinb.check("the core emits no agent_settled",
               not any(e.endswith("settled") for e in one["stream"]), "absent, as declared"),
])

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------  -------------------
PASS  the stream starts at agent_start  agent_start
PASS  tool-then-answer costs exactly two provider requests  2
PASS  the loop continued exactly while stopReason was toolUse  ['toolUse', 'stop']
PASS  the tool body ran once, and its details reached the stream  [{'words': 4}]
PASS  a model that keeps asking costs one request per ask  3
PASS  a provider failure costs one request and runs no tool  requests=1 tools=0
PASS  the core emits no agent_settled  absent, as declared

7/7 assertions held.


## Contrasting case: what a failure looks like

The third run is the interesting one. The scripted response carries
`stopReason: "error"` and an error message. The loop does **not** retry, does
**not** call the tool, and still fires `agent_end`.

That matters for the book's argument in chapter 1: `agent_end` means *the run is
over*, not *the run succeeded*. A program that watches only for `agent_end` will
treat a 429 exactly like a finished answer. Chapter 29 is about the machinery
that makes the difference visible.

## Your turn: predict, then run

**Predict first.** `keepsAsking` has two `toolUse` responses and one `stop`.
Before running the cell below, write down: how many requests, and how many tool
executions?

**Then change one input.** `TEXTS` below is what the tool is asked to count. Each
string is a separate turn.

**Predict what breaks.** The request count follows the script length, not the
words. What *would* make the count change? (Chapter 38 is about exactly that:
what happens when a run does not terminate on its own.)

In [7]:
TEXTS = ["  spaced   out  ", "one"]
import json as _json

print("TEXTS to count:", TEXTS)
print("predicted requests:", len(TEXTS) + 1, " predicted tool executions:", len(TEXTS))

again = pinb.driver(DRIVER, label="ch01-exercise", timeout=180, env={"NB_TEXTS": _json.dumps(TEXTS)})
observed = again["keepsAsking"]
print("observed requests:", observed["requests"], " tool executions:", len(observed["toolRuns"]))
print("observed word counts:", [r.get("words") for r in observed["toolRuns"]])

assert observed["requests"] == len(TEXTS) + 1
print("\nheld: the count follows the script length; the word counts are data")

TEXTS to count: ['  spaced   out  ', 'one']
predicted requests: 3  predicted tool executions: 2


observed requests: 3  tool executions: 2
observed word counts: [2, 1]

held: the count follows the script length; the word counts are data


## Interpretation: the decision you are actually writing

The loop has no "task complete" concept. It has one rule: **keep going while the
assistant's `stopReason` is `toolUse`.** Everything the chapter goes on to build
— compaction, steering, permissions, retries, the terminal — exists because that
rule needs supervising.

Two practical consequences:

1. **Termination is observable.** If you want to know whether work is finished,
   read `stopReason`. Anything else — `agent_end`, "the process exited", "it
   printed something" — is a proxy, and this notebook shows a proxy that lies.
2. **A loop with no bound is a liability.** `keepsAsking` terminated only because
   the script ran out. Chapter 38 puts a bound on the same shape and shows what
   happens when a model never submits.

### Read the ledger for this chapter

The book's own claim ledger has one row for chapter 1. Inspecting it is a useful
habit: a row names the file, the claim class and the evidence scope, so a number in
the text can be traced to the code that produced it.

In [8]:
rows = pinb.evidence_rows(1)
print(pinb.table(
    ["file", "claim", "claim class", "evidence scope", "compile", "runtime"],
    [[r["file"], r["claim"], r["claim_class"], r["evidence_scope"], r["compile"], r["runtime"]] for r in rows],
    indent="",
))

file  claim  claim class  evidence scope  compile  runtime
----------------------------------  ---------------------------------------------------------------  -----------  ------------------  -------  -------
ch01-agent-core/agent-core.test.ts  chapter 1: the agent core runs with no provider and no terminal  OBSERVED  in_process_runtime  pass  pass


## Sources, execution mode and limitations

**Execution mode:** **Run.** `examples/ch01-agent-core/agent-core.ts` executed unmodified, then a structured capture of the same loop through `drivers/ch01-loop.ts`.

**Evidence scope:** `in_process_runtime`. `@earendil-works/pi-agent-core` and `@earendil-works/pi-ai`, neither of which imports an interface package — which is what makes the chapter's "no binary, no terminal" claim checkable rather than rhetorical.

### What was read or run

- **Ran** `ch01-agent-core/agent-core.ts` — the file the chapter embeds under `<!-- example: ch01-agent-core/agent-core.ts -->`, so `npm run check-embedded` already ties it to the manuscript.
- **Ran** `drivers/ch01-loop.ts` against the pinned packages; it adds a subscriber and nothing else.
- **Read** `examples/evidence.json`, chapter 1 row.

### Limitations

- **The model is scripted.** Every branch exercised here is a branch someone wrote down in advance. Nothing here is evidence about how often a real model asks for a tool, or how often it stops when it should not.
- **No provider internals.** Retries, rate limits and context overflow are not in the core; the `error` stop reason is the whole of what the loop can see. Chapter 29.
- **`agent_settled` is absent from the core's stream.** That is a reading of the pinned declarations *and* of the stream recorded above; it is not a promise about future releases.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
